# C2 — Pipeline Supervisionado

Prepara a arquitetura do pipeline (pré-processamento + partição
persistida) sem treinar modelo. Consome o artefato do Notebook 03;
**não cria partição** e mantém o teste fechado.

**Pré-requisito:** `pip install -e .`

In [1]:
from pathlib import Path
import json
import os
import sys


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import joblib
import pandas as pd
from sklearn.base import clone

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact
from src.evaluation.leakage import check_leakage
from src.models.pipeline import build_pipeline, filtrar_rotulos_validos

env.carregar_dotenv()
env.garantir_diretorios()

print(C.resumo())

ROOT_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
DATA_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data
RAW_DIR           = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\analytical
TREINO_DIR        = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrad

## Carregar o artefato supervisionado

O artefato já traz features, alvo e coluna `particao`. `expected_kind`
precisa casar com o `cohort_kind` gravado no manifesto pelo Notebook 03.

In [ ]:
EXPECTED_KIND = "provisional"
os.environ["SRAG_COHORT_KIND"] = EXPECTED_KIND

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=EXPECTED_KIND,
    partitions=("treino", "validacao"),
)

PARTICAO  = manifest["partition_column"]
TARGET    = manifest.get("target_column") or C.cfg("target", "coluna")
FEATURES  = manifest.get("feature_columns") or C.features_finais()
PROIBIDAS = manifest.get("forbidden_columns") or list(C.cfg("features", "proibidas") or [])

assert PARTICAO in df.columns, f"partição {PARTICAO!r} ausente no artefato"
assert TARGET in df.columns, f"alvo {TARGET!r} ausente no artefato"
assert set(FEATURES).issubset(df.columns), (
    f"features ausentes: {sorted(set(FEATURES) - set(df.columns))}"
)

print(f"shape     : {df.shape}")
print(f"PARTICAO  = {PARTICAO!r}")
print(f"TARGET    = {TARGET!r}")
print(f"FEATURES  = {len(FEATURES)} colunas")
print(f"PROIBIDAS = {len(PROIBIDAS)} colunas")
df.head()

shape     : (3915132, 43)
PARTICAO  = 'particao'
TARGET    = 'alvo_obito_srag'
FEATURES  = 31 colunas
PROIBIDAS = 22 colunas


,nu_notific,grupo_split,ano_internacao,alvo_obito_srag,particao,idade_anos,dias_sintomas_ate_internacao,sexo_categoria,cs_gestant,cs_raca,...,renal,obesidade,out_morbi,sg_uf,faixa_etaria,cs_escol_n,vacina,vacina_cov,saturacao,desc_resp
0,31781201987679,31781201987679,2026,0,treino,0.083333,9.0,feminino,6,4,...,2,2,1,BA,<1,5,NaN,1,1,9
1,31773493207022,31773493207022,2026,0,teste,4.000000,1.0,feminino,6,2,...,NaN,NaN,NaN,SE,1-4,9,9,2,1,1
2,31774542694487,31774542694487,2026,0,validacao,0.030116,0.0,feminino,6,4,...,NaN,NaN,NaN,CE,<1,5,NaN,1,1,NaN
3,31773441569320,31773441569320,2026,0,treino,78.000000,0.0,feminino,6,4,...,NaN,NaN,NaN,CE,60-79,NaN,9,2,NaN,NaN
4,31783622594901,31783622594901,2026,0,treino,4.000000,4.0,feminino,6,1,...,NaN,NaN,NaN,PR,1-4,NaN,2,2,1,1


## Materializar partições

A partição já vem pronta no artefato. O teste é contado mas não aberto.

In [3]:
esperados  = {"treino", "validacao", "teste"}
observados = set(df[PARTICAO].dropna().unique())
assert observados == esperados, (
    f"partições divergentes. Esperado: {sorted(esperados)}; "
    f"Observado: {sorted(observados)}"
)

treino    = df.loc[df[PARTICAO] == "treino"].reset_index(drop=True)
validacao = df.loc[df[PARTICAO] == "validacao"].reset_index(drop=True)
n_teste   = int((df[PARTICAO] == "teste").sum())

assert len(treino) + len(validacao) + n_teste == len(df)

print(f"treino    : {len(treino):,}")
print(f"validacao : {len(validacao):,}")
print(f"teste     : {n_teste:,} (bloqueado)")

treino    : 2,740,584
validacao : 587,272
teste     : 587,276 (bloqueado)


## X / y e checagem de leakage

O `y` é lido direto da coluna do manifesto (já binária). Não
rederivamos com `criar_alvo_binario` — isso quebraria quando o alvo é
`alvo_obito_srag`. `X` usa só as features do manifesto.

In [4]:
def _alvo_do_split(particao_df):
    return pd.to_numeric(particao_df[TARGET], errors="coerce").astype("float64")


X_train, y_train = filtrar_rotulos_validos(treino[FEATURES], _alvo_do_split(treino))
X_val,   y_val   = filtrar_rotulos_validos(validacao[FEATURES], _alvo_do_split(validacao))

check_leakage(X_train, forbidden_columns=PROIBIDAS)
check_leakage(X_val,   forbidden_columns=PROIBIDAS)

print(f"treino    : {len(X_train):,}  positivos={int(y_train.sum()):,}")
print(f"validacao : {len(X_val):,}  positivos={int(y_val.sum()):,}")

treino    : 2,740,584  positivos=599,323
validacao : 587,272  positivos=128,429


## Construir o pipeline

Só o pré-processamento (imputação + escala + one-hot). O classificador
entra em C3+. Guardamos um `clone()` antes do fit como arquitetura pura.

In [5]:
pipeline = build_pipeline(X=X_train, y=y_train)
pipeline_arquitetura = clone(pipeline)

X_train_proc = pipeline.fit_transform(X_train)
X_val_proc   = pipeline.transform(X_val)

print(f"X_train: {X_train.shape} -> {X_train_proc.shape}")
print(f"X_val  : {X_val.shape} -> {X_val_proc.shape}")
pipeline

X_train: (2740584, 31) -> (2740584, 126)
X_val  : (587272, 31) -> (587272, 126)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['idade_anos','dias_sintomas_ate_internacao','sexo_categoria',...,'renal', 'obesidade','out_morbi']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature nam

## Persistir artefatos

`preprocessor.joblib` (arquitetura pura) e
`preprocessor_ajustado.joblib` (fit no treino), mais um `split_info.json`
descritivo (não recria a partição).

In [6]:
OUT_DIR = C.OUTPUTS_DIR / "C2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_path     = OUT_DIR / "preprocessor.joblib"
pipeline_fit_path = OUT_DIR / "preprocessor_ajustado.joblib"
split_info_path   = OUT_DIR / "split_info.json"

joblib.dump(pipeline_arquitetura, pipeline_path)
joblib.dump(pipeline, pipeline_fit_path)

split_info = {
    "origem":            "artefato_supervisionado",
    "cohort_kind":       manifest.get("cohort_kind"),
    "partition_column":  PARTICAO,
    "n_total":           len(df),
    "n_treino_split":    len(treino),
    "n_val_split":       len(validacao),
    "n_teste_split":     n_teste,
    "n_treino_rotulado": int(len(y_train)),
    "n_val_rotulado":    int(len(y_val)),
}
split_info_path.write_text(
    json.dumps(split_info, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Arquitetura : {pipeline_path}")
print(f"Ajustado    : {pipeline_fit_path}")
print(f"Split info  : {split_info_path}")

Arquitetura : C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\C2\preprocessor.joblib
Ajustado    : C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\C2\preprocessor_ajustado.joblib
Split info  : C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\C2\split_info.json


## Registar execução

Gravamos `metadata.json` + `summary.txt` em `outputs/runs/<RUN_ID>/`
com as features efetivamente usadas pelo `ColumnTransformer`.

In [7]:
pre          = pipeline.named_steps["pre"]
features_num = list(pre.transformers_[0][2])
features_cat = list(pre.transformers_[1][2])

with Run(modelo="C2_pipeline", notas="Arquitetura do pipeline supervisionado") as r:
    r.add_dataset(
        n_rows=len(df),
        train_rows=int(len(y_train)),
        valid_rows=int(len(y_val)),
        test_rows=n_teste,
    )
    r.features = {
        "n_features":  len(features_num) + len(features_cat),
        "numericas":   features_num,
        "categoricas": features_cat,
    }
    r.parametros = {
        "target":           TARGET,
        "partition_column": PARTICAO,
        "origem_particao":  "artefato_supervisionado",
        "cohort_kind":      manifest.get("cohort_kind"),
    }
    r.metrica(n_features_processadas=int(X_train_proc.shape[1]))

print("Run registrado em:", r.pasta)

[reproducibility] run registrado em: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\runs\2026-10-16_215802_867822_C2_pipeline
Run registrado em: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\runs\2026-10-16_215802_867822_C2_pipeline
